In [2]:
import os
import glob
import re
import numpy as np
import xml.etree.ElementTree as ET

# 1. EXACT SAME SORTING LOGIC AS YOUR SPATIAL PIPELINE
def parse_camera_and_frame(filename):
    """Extracts Camera ID and Frame Number, forcing the exact order of the feature matrix."""
    import os
    import re
    base_name = os.path.basename(filename)
    match = re.search(r'(Cam\d+).*?_frame_(\d+)\.', base_name)
    if match:
        cam_id = match.group(1)
        frame_num = int(match.group(2))
        
        # Force the exact order generated by spatial_pipeline.ipynb
        order_dict = {'Cam7': 0, 'Cam1': 1, 'Cam5': 2}
        cam_order = order_dict.get(cam_id, 99)
        
        # Sorts by Camera Order first, then Frame Number chronologically
        return cam_order, frame_num
        
    return 99, 0

# Replace this with the path to your XML test files
xml_folder_path = 'Images/*.xml' 
xml_paths = glob.glob(xml_folder_path)

# Sort the files strictly chronologically per camera
xml_paths.sort(key=parse_camera_and_frame)

print(f"Total XML files found and sorted: {len(xml_paths)}")

# 2. EXTRACT LABELS STRICTLY FOR WEAPONS
raw_labels = []
weapon_classes = ['Handgun', 'Short_rifle', 'Knife']

for xml_file in xml_paths:
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    is_anomaly = 0
    for obj in root.findall('object'):
        name = obj.find('name').text
        # ONLY flag as 1 if the object is explicitly a weapon
        if name in weapon_classes:
            is_anomaly = 1
            break
            
    raw_labels.append(is_anomaly)

# 3. SEQUENCE ENGINEERING (Sliding Window & Camera Borders)
# We must drop the 9 boundary frames per camera just like the CNN pipeline did.
# Cam7: 3511 frames -> drops 9
# Cam1: 607 frames -> drops 9
# Cam5: 1031 frames -> drops 9

camera_limits = [3511, 607, 1031]
sequence_length = 10
final_sequence_labels = []
start_idx = 0

for limit in camera_limits:
    end_idx = start_idx + limit
    camera_block = raw_labels[start_idx:end_idx]
    
    # Slide the 10-frame window
    for i in range(0, len(camera_block) - sequence_length + 1):
        window = camera_block[i:i + sequence_length]
        # If ANY frame in the 5-second window has a weapon, the whole sequence is a threat
        if 1 in window:
            final_sequence_labels.append(1)
        else:
            final_sequence_labels.append(0)
            
    start_idx = end_idx

# 4. EXPORT
y_true_binary = np.array(final_sequence_labels)
np.save('y_true_binary.npy', y_true_binary)

print(f"Perfect Chronological Labels Exported: {y_true_binary.shape[0]} sequences.")
print(f"Total True Normal (0): {np.sum(y_true_binary == 0)}")
print(f"Total True Anomaly (1): {np.sum(y_true_binary == 1)}")

Total XML files found and sorted: 5149
Perfect Chronological Labels Exported: 5122 sequences.
Total True Normal (0): 2496
Total True Anomaly (1): 2626
